# RAGAS
To evaluate answer quality, I am going to generate question answer pairs using openai api


In [5]:
# Cell 1: Imports and setup
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import random
random.seed(69)



In [ ]:
load_dotenv()

PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"

In [4]:
# Loads  chunks as LangChain documents
# Using the already-chunked data rather than raw PDFs since RAGAS should generate questions from what the pipeline actually retrieves



embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma(
    persist_directory=str(DATA_DIR / "chromadb"),
    collection_name="recursive_100",#typo of 1000
    embedding_function=embedding_model
)

# Get all documents from the collection
all_docs = vectorstore.get()
print(f"Loaded {len(all_docs['documents'])} chunks")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4856.67it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 31741 chunks


In [ ]:
# Convert sto LangChain Documents for RAGAS
docs = []
for i, (text, metadata) in enumerate(zip(all_docs['documents'], all_docs['metadatas'])):
    docs.append(Document(page_content=text, metadata=metadata))

# Samples a subset; 31k chunks is way too many, RAGAS doesn't need all of them

docs_sample = random.sample(docs, min(200, len(docs)))
print(f"Sampled {len(docs_sample)} chunks for test generation")

Sampled 200 chunks for test generation


In [ ]:
#refactor
from openai import OpenAI; from ragas.llms import llm_factory;
#llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))

In [ ]:

#Sets up RAGAS generator to run overnight
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))#using best available model for a good price
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings
)

testset = generator.generate_with_langchain_docs(
    docs_sample,
    testset_size=50
)

# Saves results
df = testset.to_pandas()
df.to_csv(PROJECT_ROOT / "outputs" / "ragas_testset.csv", index=False)
print(f"Generated {len(df)} test cases")
df.head()

<positron-console-cell-9>:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
<positron-console-cell-9>:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
Applying CustomNodeFilter:  18%|█▊        | 37/200 [00:24<01:26,  1.89it/s]Node 83b12662-245e-45de-844b-296a4aea034a does not have a summary. Skipping filtering.
Node 69597741-3d38-4df6-8852-ca8563d88739 does not have a summary. Skipping filtering.
Node 5ab2bc5f-97e9-4f3c-bec4-45ea81af96a0 does not have a summary. Skipping filtering.
Node 997119d0-4aea-41c1-94a1-a8775a7efcf3 

Generated 51 test cases


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Can you explain the significance of Scholz et ...,[steps for functional annotation are similar. ...,Scholz et al. 2012 is significant in the conte...,Marine Ecologist,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,What research did Siegelman H.W. contribute to...,"[Paerl H.W., Bland P.T., Bowles N.D. and Haiba...",Siegelman H.W. co-authored a study in 1983 tha...,Estuarine Ecologist,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
2,How is the AB28 primer utilized in the amplifi...,[rbcL-rbcS intergenic spacer to the first codo...,The AB28 primer is used to amplify the ITS1-5....,Estuarine Ecologist,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
3,What is Raphidiopsis mediterranea Skuja?,"[Miller (Pl. 2, Figs 5-8) . Geitler, 1925, p. ...",Raphidiopsis mediterranea Skuja is a species c...,Estuarine Ecologist,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
4,What distinguishes Centroceras hommersandii fr...,[Diagnosis. Differing from other species in th...,Centroceras hommersandii is distinguished from...,Marine Ecologist,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer


In [12]:
df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Can you explain the significance of Scholz et ...,[steps for functional annotation are similar. ...,Scholz et al. 2012 is significant in the conte...,Marine Ecologist,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,What research did Siegelman H.W. contribute to...,"[Paerl H.W., Bland P.T., Bowles N.D. and Haiba...",Siegelman H.W. co-authored a study in 1983 tha...,Estuarine Ecologist,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
2,How is the AB28 primer utilized in the amplifi...,[rbcL-rbcS intergenic spacer to the first codo...,The AB28 primer is used to amplify the ITS1-5....,Estuarine Ecologist,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
3,What is Raphidiopsis mediterranea Skuja?,"[Miller (Pl. 2, Figs 5-8) . Geitler, 1925, p. ...",Raphidiopsis mediterranea Skuja is a species c...,Estuarine Ecologist,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
4,What distinguishes Centroceras hommersandii fr...,[Diagnosis. Differing from other species in th...,Centroceras hommersandii is distinguished from...,Marine Ecologist,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
